In [4]:
import sys
sys.path.append('../')

import numpy as np
from matplotlib import pyplot as plt
from qutip.qip.operations import rz, cz_gate
from tqdm import tqdm
from matplotlib.colors import LogNorm
import pytz, cmath, itertools
import scqubits.settings as settings
settings.OVERLAP_THRESHOLD = 0.3
from joblib import Parallel, delayed
import scipy.sparse as ssp
from sympy import symbols
import utils_2Q_gate_zp as ut
import pandas as pd
import scipy as sp
from multiprocessing import Pool
import qutip as qt
import multiprocessing as mp
from multiprocessing import Pool
import scqubits as scq
from sympy import symbols
import scipy.sparse as ssp
from datetime import datetime

In [3]:
qt.propagator?

Signature:
qt.propagator(
    H,
    t,
    c_op_list=[],
    args={},
    options=None,
    unitary_mode='batch',
    parallel=False,
    progress_bar=None,
    _safe_mode=True,
    **kwargs,
)
Docstring:
Calculate the propagator U(t) for the density matrix or wave function such
that :math:`\psi(t) = U(t)\psi(0)` or
:math:`\rho_{\mathrm vec}(t) = U(t) \rho_{\mathrm vec}(0)`
where :math:`\rho_{\mathrm vec}` is the vector representation of the
density matrix.

Parameters
----------
H : qobj or list
    Hamiltonian as a Qobj instance of a nested list of Qobjs and
    coefficients in the list-string or list-function format for
    time-dependent Hamiltonians (see description in :func:`qutip.mesolve`).

t : float or array-like
    Time or list of times for which to evaluate the propagator.

c_op_list : list
    List of qobj collapse operators.

args : list/array/dictionary
    Parameters to callback functions for time-dependent Hamiltonians and
    collapse operators.

options : :class:`qu

In [5]:
data = np.load('data/collapse_ops_n=100_170us.npz')
np.shape(data['c_op_list'])

(24023, 100, 100)

In [ ]:
def load_simulation_data(truc_full, folder = '../../data/3ncut_two_zeropi/truc1=300_truc2=1000_pick=True/'):
    """
    Loads the energy spectrum and matrix elements (n_theta, n_phi) for the 0-π qubit.
    The function "generate_data()" in sigmaX_fidelity_import_paras.py can generate the data
    """    
    hspace_0 = pd.read_csv(folder+ 'hspace_0.txt').to_numpy().flatten()
    hspace_1 = pd.read_csv(folder+ 'hspace_1.txt').to_numpy().flatten()
    hspace_full = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()[:truc_full]
    eket_tot = ssp.csr_matrix(np.load(folder+ 'eket_tot.npy'))[:truc_full]
    eval_tot = 2*np.pi* pd.read_csv(folder+ 'eval_tot.txt').to_numpy().flatten()[:truc_full]
    n_theta0_dress = 2*np.pi* np.load(folder+'n_theta0_dress.npy')
    n_theta1_dress = 2*np.pi* np.load(folder+'n_theta1_dress.npy')
    dim_0 = len(hspace_0)
    dim_1 = len(hspace_1)
    n_theta0_dress = ut.truncate_2(n_theta0_dress, np.arange(truc_full))
    n_theta1_dress = ut.truncate_2(n_theta1_dress, np.arange(truc_full))

    return hspace_full, eket_tot, eval_tot, n_theta0_dress, n_theta1_dress, dim_0, dim_1

def load_drive_params():
    """
    Load CZ-gate drive parameters from a CSV file.
    """
    cz300_se_3ncut= pd.read_csv('data/data_cz_3ncut_truc1=300_select.txt')
    x0_vec = cz300_se_3ncut[['tg', 'drive_amp', 'detune']].to_numpy()
    return x0_vec

def build_hamiltonian_given_states(hspace_full, hspace_select, eket_tot, eval_tot, drive_term):
    """
    Constructs the truncated Hamiltonian and drive terms.
    """
    logi_state = ['0-0', '0-2', '2-0', '2-2']
    index_select = [hspace_full.index(i) for i in hspace_select]
    H0_full = qt.Qobj(np.diag(eval_tot))
    H0_select = ut.truncate_2( H0_full, index_select)
    drive_select = ut.truncate_2(drive_term, index_select)
    eket_tot = eket_tot[index_select]
    logi_idx_select = [hspace_select.index(i) for i in logi_state]
    H_drive_select = [ H0_select,   [drive_select, ut.drive_gauss_A] ]
    return H_drive_select, logi_idx_select, eket_tot, index_select

def build_hamiltonian_by_energy(hspace_full, n_hspace, eket_tot, eval_tot, drive_term):
    """
    Constructs the truncated Hamiltonian and drive terms.
    """
    logi_state = ['0-0', '0-2', '2-0', '2-2']
    index_select = np.arange(n_hspace)
    hspace_select = hspace_full[:n_hspace]
    
    H0_full = qt.Qobj(np.diag(eval_tot))
    H0_select = ut.truncate_2( H0_full, index_select)
    drive_select = ut.truncate_2(drive_term, index_select)
    eket_tot = eket_tot[index_select]
    logi_idx_select = [hspace_select.index(i) for i in logi_state]
    H_drive_select = [ H0_select,   [drive_select, ut.drive_gauss_A] ]
    return H_drive_select, logi_idx_select, eket_tot, index_select


def construct_c_ops(dim_0, dim_1, n_theta0, n_theta1, gamma_dephase_02_q0, gamma_dephase_02_q1, eket_tot):
    """
    Constructs collapse operators for dissipation.
    """
    qubit_a = True
    arg_a = [dim_0, dim_1, n_theta0, eket_tot, qubit_a] 
    state_vec_a = [(i,j) for i in range(dim_0) for j in range(dim_0) if i < j]
    jump_t1_a = Parallel(n_jobs=10)(delayed(ut.get_jump_op_decay)(state_vec, *arg_a) for state_vec in state_vec_a)

    arg_a = [dim_0, dim_1, gamma_dephase_02_q0, eket_tot, qubit_a] 
    jump_tphi_a = Parallel(n_jobs=10)(delayed(ut.get_jump_op_dephase)(state_j, *arg_a) for state_j in range(1,dim_0))

    qubit_a = False
    arg_b = [dim_0, dim_1, n_theta1, eket_tot, qubit_a] 
    state_vec_b = [(i,j) for i in range(dim_1) for j in range(dim_1) if i < j]
    jump_t1_b = Parallel(n_jobs=10)(delayed(ut.get_jump_op_decay)(state_vec, *arg_b) for state_vec in state_vec_b)

    arg_b = [dim_0, dim_1, gamma_dephase_02_q1, eket_tot, qubit_a] 
    jump_tphi_b = Parallel(n_jobs=10)(delayed(ut.get_jump_op_dephase)(state_j, *arg_b) for state_j in range(1,dim_1))

    return jump_t1_a + jump_tphi_a + jump_t1_b + jump_tphi_b

def load_noise_data(t1_tphi_other, folder = '../../data/3ncut_two_zeropi/truc1=500/'):
    """
    Load dephasing rates calculated for 50μs.    
    """    
    gamma_q0 = pd.read_csv(folder+ 'data_gamma_qubit0.txt')
    gamma_q1 = pd.read_csv(folder+ 'data_gamma_qubit1.txt')
    n_theta0 = np.load(folder+'n_theta0.npy')
    n_theta1 = np.load(folder+'n_theta1.npy')
    gamma_dephase_02_q0 = gamma_q0['tphi_02'].to_numpy() *50 /t1_tphi_other
    gamma_dephase_02_q1 = gamma_q1['tphi_02'].to_numpy() *50 /t1_tphi_other
    return n_theta0, n_theta1, gamma_dephase_02_q0, gamma_dephase_02_q1


In [ ]:
calculate_ideal, calculate_noise =  True, True # False, True  # whether to calculate noisy fidelity
t1_tphi_other = 170 # μs
truc_full = 500
num_cpus, n_job = 16, 10
tg_list = [0] # [2,  9, 16, 23, 30]  # Select the first row for testing

max_step_ideal = 1e-4 # Set max_step to 0 for parallel execution
nsteps_ideal = 1/ max_step_ideal  # Set nsteps to a large number for parallel execution
max_step_noisy = 1e-4 # Set max_step to 0 for parallel execution
nsteps_noisy = 1/ max_step_noisy  # Set nsteps to a large number for parallel execution
hspace_select = [ ### state_all_1000 ### charge_pick
'0-0', '5-0', '0-2', '2-0', '2-2', '5-2', '5-1', '0-1', '2-1', '1-0' ,
'0-5', '2-5', '1-2', '9-0', '4-0', '2-4', '5-5', '1-1', '5-4', '1-5' ,
'9-2', '0-4', '4-2', '2-8', '0-8', '9-1', '2-12', '2-9', '0-9', '0-12' ,
'12-0', '2-16', '0-16', '5-8', '1-8', '4-5', '2-13', '2-21', '0-18', '2-20' ,
'9-4', '4-9', '8-0', '2-18', '0-21', '2-24', '1-4', '18-0', '5-26', '0-13' ,

'13-0', '0-26', '15-0', '2-26', '1-12', '5-16', '8-1', '0-24', '15-1', '2-35' ,
'15-4', '2-33', '2-45', '0-33', '2-39', '0-45', '5-12', '0-39', '5-9', '8-12' ,
'5-33', '12-2', '4-4', '1-9', '4-1', '5-34', '2-30', '2-46', '1-16', '0-34' ,
'2-34', '8-2', '5-21', '2-52', '0-52', '0-42', '2-42', '2-59', '0-59', '5-18' ,
'0-65', '5-24', '2-55', '0-55', '0-20', '9-8', '8-9', '22-0', '1-25', '8-5' ,

'12-1', '4-8', '2-53', '2-36', '2-25', '5-20', '5-13', '9-24', '15-8', '1-30' ,
'0-25', '1-20', '5-25', '9-5', '1-13', '1-24', '13-2', '1-33', '18-1', '0-68' ,
'18-2', '20-0', '2-44', '9-12', '4-25', '9-9', '5-30', '0-83', '5-39', '9-16' ,
'0-36', '0-73', '12-4', '18-5', '22-2', '15-16', '1-21', '5-35', '9-13', '2-60' ,
'2-57', '15-2', '15-5', '2-28', '13-1', '4-12', '0-35', '9-20', '2-54', '1-18' ,

'0-81', '24-1', '4-39', '0-30', '1-26', '8-4', '4-16', '12-5', '1-35', '8-8' ,
'24-0', '12-9', '5-44', '0-77', '4-21', '0-57', '5-28', '1-39', '25-0', '8-18' ,
'1-28', '4-44', '33-0', '5-42', '12-12', '0-54', '13-12', '9-26', '4-24', '20-9' ,
'8-26', '24-4', '8-16', '0-46', '13-4', '1-34', '37-1', '0-44', '18-16', '22-4' ,
'1-45', '0-78', '9-25', '5-36', '24-9', '1-44', '4-35', '18-8', '0-53', '1-60' ,
]

option_ideal =qt.Options(max_step=max_step_ideal, nsteps=nsteps_ideal, num_cpus=1)  
option_noisy =qt.Options(max_step=max_step_noisy, nsteps=nsteps_noisy, num_cpus=1)  

hspace_full, eket_tot, eval_tot, n_theta0_dress, n_theta1_dress, dim_0, dim_1 = load_simulation_data(truc_full)


: 

In [ ]:
params = load_drive_params()[tg_list, ]  # [1::4,] # Load pulse parameters from CSV
W_20_50 = eval_tot[hspace_full.index('5-0')] - eval_tot[hspace_full.index('2-0')]
drive_term = n_theta1_dress

len_select = len(hspace_select)
H_drive_select, logi_idx_select, eket_tot, index_select = build_hamiltonian_given_states(hspace_full, hspace_select, eket_tot, eval_tot, drive_term)
print("t1_tphi_other = ", t1_tphi_other)
print('truc_full=', truc_full )
print('num_cpus=', num_cpus, ', n_job=', n_job)
print(f"calculate_ideal = {calculate_ideal}, calculate_noise = {calculate_noise}")
print(f'Ideal: max_step = {option_ideal.max_step}, nsteps = {option_ideal.nsteps}')
print(f'Noisy: max_step = {option_noisy.max_step}, nsteps = {option_noisy.nsteps}')
ut.print_data_r2r(f'params', params.tolist(), num_each_row=1)
ut.print_data_r2r(f'hspace_select (len={len_select})', hspace_select, num_each_row=10)
ut.print_data_r2r(f'index_select (len={len_select})', index_select, num_each_row=10)

if calculate_ideal: # ideal fidelity
    c_op_list = []
    arg_select = [H_drive_select, W_20_50, num_cpus, c_op_list, logi_idx_select, option_ideal, option_noisy]
    f_ideal = Parallel(n_jobs=n_job)(delayed(ut.cz_fidelity_log_noise)(args_indep, *arg_select)
                                                for args_indep in params)
    ut.print_data_r2r(f'f_ideal_{len_select}', f_ideal)
    ut.print_time()

if calculate_noise: # Noisey fidelity
    n_theta0, n_theta1, gamma_dephase_02_q0, gamma_dephase_02_q1 = load_noise_data(t1_tphi_other)
    c_op_list = construct_c_ops(dim_0, dim_1, n_theta0, n_theta1, gamma_dephase_02_q0, 
                                gamma_dephase_02_q1, eket_tot) # Construct collapse operators
    arg_select = [H_drive_select, W_20_50, num_cpus, c_op_list, logi_idx_select, option_ideal, option_noisy]
    f_noise = Parallel(n_jobs=n_job)(delayed(ut.cz_fidelity_log_noise)(args_indep, *arg_select)
                                                for args_indep in params)
    ut.print_data_r2r(f'f_{t1_tphi_other}us_{len_select}', f_noise)






# Coupled zero pi

In [27]:
truc_full = 500
num_cpus, n_job = 16, 10
folder = f'../../data/3ncut_two_zeropi/truc1=300_truc2=1000_pick=True/'
hspace_0 = pd.read_csv(folder+ 'hspace_0.txt').to_numpy().flatten()
hspace_1 = pd.read_csv(folder+ 'hspace_1.txt').to_numpy().flatten()
hspace_full = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()[:truc_full]
eket_tot = ssp.csr_matrix(np.load(folder+ 'eket_tot.npy'))[:truc_full]
eval_tot = 2*np.pi* pd.read_csv(folder+ 'eval_tot.txt').to_numpy().flatten()[:truc_full]
n_theta0_dress = 2*np.pi* np.load(folder+'n_theta0_dress.npy')
n_theta1_dress = 2*np.pi* np.load(folder+'n_theta1_dress.npy')
dim_0 = len(hspace_0)
dim_1 = len(hspace_1)
n_theta0_dress = ut.truncate_2(n_theta0_dress, np.arange(truc_full))
n_theta1_dress = ut.truncate_2(n_theta1_dress, np.arange(truc_full))

### cz part
cz300_se_3ncut= pd.read_csv('data/data_cz_3ncut_truc1=300_select.txt')
x0_vec = cz300_se_3ncut[['tg', 'drive_amp', 'detune']].to_numpy()[[0],:]
# [[-4],:]   [0::2,:] [[0,3,4, 16],:]

W_20_50 = eval_tot[hspace_full.index('5-0')] - eval_tot[hspace_full.index('2-0')]
drive_term = n_theta1_dress
hspace_select = [
    ### state_all_1000
    ### charge_pick
'0-0', '0-1', '1-0', '0-2', '2-0', '0-4', '4-0', '1-1', '0-5', '2-1' ,
'5-0', '1-2', '0-8', '2-2', '8-0', '1-4', '4-1', '0-9', '2-4', '9-0' ,
# '1-5', '5-1', '4-2', '0-12', '2-5', '1-8', '12-0', '5-2', '0-13', '0-16' ,
# '8-1', '4-4', '13-0', '15-0', '2-8', '1-9', '9-1', '8-2', '5-4', '4-5' ,
# '0-18', '2-9', '1-12', '0-20', '0-21',
# '18-0', '9-2', '12-1', '5-5', '0-24' ,

# '4-8', '1-13', '20-0', '8-4', '1-16',
# '22-0', '2-12', '13-1', '24-0', '0-25' ,
# '15-1', '0-26', '5-8', '4-9', '12-2', '2-13', '9-4', '2-16', '8-5', '25-0' ,
# '13-2', '26-0', '1-18', '15-2', '0-28', '5-9', '4-12', '28-0', '0-30', '18-1' ,
# '12-4', '9-5', '1-20', '1-21', '0-33',
# '8-8', '0-34', '0-35', '1-24', '2-18' ,
# '0-36', '20-1', '4-13', '22-1', '4-16',
# '30-0', '13-4', '5-12', '15-4', '24-1' ,
]

### common part
logi_state = ['0-0', '0-2', '2-0', '2-2']
index_select = [hspace_full.index(i) for i in hspace_select]
len_select = len(hspace_select)
H0_full = qt.Qobj(np.diag(eval_tot))
H0_select = ut.truncate_2( H0_full, index_select)
drive_select = ut.truncate_2(drive_term, index_select)
eket_tot = eket_tot[index_select]
logi_idx_select = [hspace_select.index(i) for i in logi_state]
H_drive_select = [ H0_select,   [drive_select, ut.drive_gauss_A] ]
print('truc_full=', truc_full )
print('num_cpus=', num_cpus, ', n_job=', n_job)
print('params =')
for para in x0_vec:
    print(para.tolist(), ',')
print(f'\nhspace_select (len={len_select}) = [')
for i in range(0, len(hspace_select), 10):
    print(", ".join(f"'{x}'" for x in hspace_select[i:i + 10]), ',')
print(']')
states_all_index = [hspace_full.index(i) for i in hspace_select]
data = states_all_index
print(f'\nhspace_select_index = [')
for i in range(0, len(data), 10):  # Step size of 10
    print(", ".join(f"{x}" for x in data[i:i + 10]), ',')
print(']')


truc_full= 500
num_cpus= 16 , n_job= 10
params =
[20.032421, 0.045974, 0.029778] ,

hspace_select (len=20) = [
'0-0', '0-1', '1-0', '0-2', '2-0', '0-4', '4-0', '1-1', '0-5', '2-1' ,
'5-0', '1-2', '0-8', '2-2', '8-0', '1-4', '4-1', '0-9', '2-4', '9-0' ,
]

hspace_select_index = [
0, 1, 2, 3, 4, 5, 6, 7, 8, 9 ,
10, 11, 12, 13, 14, 15, 16, 17, 18, 19 ,
]


In [29]:
t1 = 170    
gamma = 1 / 1e3 / t1 # calculate decay rate given T1, unit in micro-second

folder = f'../../data/3ncut_two_zeropi/truc1=500/'
n_theta0 = np.load(folder+'n_theta0.npy')
n_theta1 = np.load(folder+'n_theta1.npy')
Gamma_decay_0 = gamma / (np.abs(n_theta0[4,8])**2)
Gamma_decay_1 = gamma / (np.abs(n_theta1[4,8])**2)
gamma_decay_0 = Gamma_decay_0 * np.abs(n_theta0) ** 2
gamma_decay_1 = Gamma_decay_1 * np.abs(n_theta1) ** 2

folder = f'../../data/3ncut_two_zeropi/truc1=500/'
gamma_q0 = pd.read_csv(folder+ 'data_gamma_qubit0.txt')
gamma_q1 = pd.read_csv(folder+ 'data_gamma_qubit1.txt')
gamma_dephase_0 = gamma_q0['tphi_02'].to_numpy() *50 /t1
gamma_dephase_1 = gamma_q1['tphi_02'].to_numpy() *50 /t1


In [30]:
jump_t1 = []
jump_tphi = []
for i in range(1, len(hspace_select)): # take |1,2><3,4| as an example: '1-2'--'3,4'
    si_1, si_2 = map(int, hspace_select[i].split('-')) # extract '1' and '2' for state '1-2'
    for j in range(i):
        sj_1, sj_2 = map(int, hspace_select[j].split('-')) # extract '3' and '4' for state '3-4'
        # t_1
        jop_0 = np.sqrt(gamma_decay_0[si_1, sj_1])* qt.basis(dim_0, si_1) * qt.basis(dim_0, sj_1).dag() # |1><3| for qubit 1
        jop_1 = np.sqrt(gamma_decay_1[si_2, sj_2])* qt.basis(dim_1, si_2) * qt.basis(dim_1, sj_2).dag() # |2><4| for qubit 2
        jop = qt.tensor(jop_0, jop_1) # get tensored jump op
        jump_t1.append(qt.Qobj( eket_tot @ jop.data @ eket_tot.conj().T ))
    # t_phi
    proj_0 = np.sqrt(2*gamma_dephase_0[si_1] )* qt.basis(dim_0, si_1).proj()
    proj_1 = np.sqrt(2*gamma_dephase_1[si_2] )* qt.basis(dim_1, si_2).proj()
    proj = qt.tensor(proj_0, proj_1)
    jump_tphi.append(qt.Qobj( eket_tot @ proj.data @ eket_tot.conj().T))        

In [31]:
#################################################################
### Noisey fidelity
t1_tphi_other = 170 # μs

print("t1_tphi_other = ", t1_tphi_other)
folder = f'../../data/3ncut_two_zeropi/truc1=500/'
gamma_q0 = pd.read_csv(folder+ 'data_gamma_qubit0.txt')
gamma_q1 = pd.read_csv(folder+ 'data_gamma_qubit1.txt')
gamma_decay_48_q0 = gamma_q0['t1_50us_48'].to_numpy() *50 /t1_tphi_other
gamma_decay_48_q1 = gamma_q1['t1_50us_48'].to_numpy() *50 /t1_tphi_other
gamma_dephase_02_q0 = gamma_q0['tphi_02'].to_numpy() *50 /t1_tphi_other
gamma_dephase_02_q1 = gamma_q1['tphi_02'].to_numpy() *50 /t1_tphi_other

qubit_a = True
arg_a = [dim_0, dim_1, gamma_decay_48_q0, gamma_dephase_02_q0, eket_tot, qubit_a] # old gamma
jump_op_a = Parallel(n_jobs=100)(delayed(ut.get_jump_op_charge_pick)(state, *arg_a) for state in range(1,dim_0))

# qubit_a = False
# arg_b = [dim_0, dim_1, gamma_decay_48_q1, gamma_dephase_02_q1, eket_tot, qubit_a] # old gamma
# jump_op_b = Parallel(n_jobs=100)(delayed(ut.get_jump_op_charge_pick)(state, *arg_b) for state in range(1,dim_1))

# jump_t1_list = np.array(jump_op_a)[:,0].tolist() + np.array(jump_op_b)[:,0].tolist()
# jump_tphi_list = np.array(jump_op_a)[:,1].tolist() + np.array(jump_op_b)[:,1].tolist()

# jump_t1_list = [qt.Qobj(matrix) for matrix in jump_t1_list]
# jump_tphi_list = [qt.Qobj(matrix) for matrix in jump_tphi_list]


t1_tphi_other =  170


In [ ]:
def get_jump_op(state_vec, *args):
    state_i, state_j = state_vec
    dim_0, dim_1, n_theta, gamma_dephase, eket_tot, qubit_a = args
    if qubit_a:
        # t_1
        ladder_ij = qt.basis(dim_0, state_i) * qt.basis(dim_0, state_j).dag()
        a_ij_I = qt.tensor(ladder_ij, qt.qeye(dim_1))
        jump_t1 = qt.Qobj( eket_tot @ ( np.sqrt(abs(n_theta[state_i, state_j]))* a_ij_I ).data @ eket_tot.conj().T )
        # t_phi
        proj_jj = qt.basis(dim_0, state_j).proj()
        a_jj_I = qt.tensor(proj_jj, qt.qeye(dim_1))
        jump_tphi = qt.Qobj( eket_tot @ ( np.sqrt(2*gamma_dephase[state_j] )* a_jj_I  ).data @ eket_tot.conj().T)
    else: # qubit_b
        # t_1
        ladder_ij = qt.basis(dim_1, state_i) * qt.basis(dim_1, state_j).dag()
        a_I_ij = qt.tensor(qt.qeye(dim_0), ladder_ij)
        jump_t1 = qt.Qobj( eket_tot @ ( np.sqrt(abs(n_theta[state_i, state_j]))* a_I_ij ).data @ eket_tot.conj().T )
        # t_phi
        proj_jj = qt.basis(dim_1, state_j).proj()
        a_I_jj = qt.tensor(qt.qeye(dim_0), proj_jj)
        jump_tphi = qt.Qobj( eket_tot @ ( np.sqrt(2*gamma_dephase[state_j] )* a_I_jj  ).data @ eket_tot.conj().T)
    return [jump_t1, jump_tphi]

def get_jump_op_decay(state_vec, *args):
    state_i, state_j = state_vec
    dim_0, dim_1, n_theta, eket_tot, qubit_a = args
    if qubit_a:
        ladder_ij = qt.basis(dim_0, state_i) * qt.basis(dim_0, state_j).dag()
        jump_t1 = np.sqrt(abs(n_theta[state_i, state_j])) * qt.tensor(ladder_ij, qt.qeye(dim_1))
    else: # qubit_b
        ladder_ij = qt.basis(dim_1, state_i) * qt.basis(dim_1, state_j).dag()
        jump_t1 = np.sqrt(abs(n_theta[state_i, state_j])) * qt.tensor(qt.qeye(dim_0), ladder_ij)
    jump_t1 = qt.Qobj( eket_tot @ ( np.sqrt(abs(n_theta[state_i, state_j])) * jump_t1 ).data @ eket_tot.conj().T )
    return jump_t1

def get_jump_op_dephase(state_j, *args):
    dim_0, dim_1, gamma_dephase, eket_tot, qubit_a = args
    if qubit_a:
        proj_jj = qt.basis(dim_0, state_j).proj()
        jump_tphi = np.sqrt( 2*gamma_dephase[state_j] ) * qt.tensor(proj_jj, qt.qeye(dim_1))
    else: # qubit_b
        proj_jj = qt.basis(dim_1, state_j).proj()
        jump_tphi = np.sqrt( 2*gamma_dephase[state_j] ) * qt.tensor(qt.qeye(dim_0), proj_jj)
    jump_tphi = qt.Qobj( eket_tot @ jump_tphi.data @ eket_tot.conj().T)
    return jump_tphi

qubit_a = True
arg_a = [dim_0, dim_1, n_theta0, eket_tot, qubit_a] 
state_vec_a = [(i,j) for i in range(dim_0) for j in range(dim_0) if i < j]
jump_t1_a = Parallel(n_jobs=10)(delayed(get_jump_op_decay)(state_vec, *arg_a) for state_vec in state_vec_a)

arg_a = [dim_0, dim_1, gamma_dephase_02_q0, eket_tot, qubit_a] 
jump_tphi_a = Parallel(n_jobs=10)(delayed(get_jump_op_dephase)(state_j, *arg_a) for state_j in range(1,dim_0))

qubit_a = False
arg_b = [dim_0, dim_1, n_theta1, eket_tot, qubit_a] 
state_vec_b = [(i,j) for i in range(dim_1) for j in range(dim_1) if i < j]
jump_t1_b = Parallel(n_jobs=10)(delayed(get_jump_op_decay)(state_vec, *arg_b) for state_vec in state_vec_b)

arg_b = [dim_0, dim_1, gamma_dephase_02_q1, eket_tot, qubit_a] # old gamma
jump_tphi_b = Parallel(n_jobs=10)(delayed(get_jump_op_dephase)(state_j, *arg_b) for state_j in range(1,dim_1))

In [69]:
jump_t1_a[0]

Quantum object: dims = [[20], [20]], shape = (20, 20), type = oper, isherm = False
Qobj data =
[[-5.67893138e-16+1.11600374e-15j  2.98617422e-01-2.94765691e-01j
  -9.36934670e-01-8.79679355e-01j  1.72743299e-15-8.46956540e-16j
  -2.50150725e-15+3.88057216e-16j -2.21275074e-15+2.74490907e-15j
  -3.80104926e-16+1.04024028e-15j -9.00539855e-16+1.29912281e-15j
   7.85908797e-05+1.11482495e-04j -9.11760681e-05+7.02008996e-05j
  -3.39410639e-05-1.69721568e-04j  1.56502787e-05-4.56628314e-05j
  -7.22075016e-04+1.05444591e-03j -3.18289352e-16+2.98581430e-16j
   7.84282268e-03+7.60006289e-03j -5.44770581e-04+8.48959746e-05j
   1.40605121e-02-1.10289096e-02j -1.37970048e-15+1.04636783e-15j
  -3.60610759e-18+7.54347729e-17j  6.72011258e-17-4.67721324e-17j]
 [ 1.00423626e-02+9.91283073e-03j  7.90668612e-16+1.11946434e-15j
   1.61881769e-15+7.81265767e-16j  2.63433090e-04-5.89174994e-04j
  -8.55532359e-05+3.43054733e-04j  1.85558663e-01+2.64229000e-01j
   6.32280207e-02-8.62426976e-01j  4.64623568e

In [ ]:
# np.savez('data_file.npz', jump_op_a=jump_op_a, jump_op_b=jump_op_b)

In [3]:
### cz part
# ### ideal fidelity
# c_op_list = [qt.Qobj(np.zeros((len_select, len_select)))]
c_op_list = []
arg_select = [H_drive_select, W_20_50, num_cpus, c_op_list, logi_idx_select]
f_ideal = Parallel(n_jobs=n_job)(delayed(ut.cz_fidelity_log_optimize)(args_indep, *arg_select)
                                            for args_indep in x0_vec)
print(f'\nf_ideal (dim={len(hspace_select)})  = [')
for i in range(0, len(f_ideal), 4):
    print(', '.join(map(str, np.round(f_ideal[i:i+4], 8).tolist())), ',')
print(']')


f_ideal (dim=100)  = [
-0.68353533 ,
]


In [ ]:

c_op_list = jump_t1_list + jump_tphi_list
arg_select = [H_drive_select, W_20_50, num_cpus, c_op_list, logi_idx_select]
f_noise = Parallel(n_jobs=n_job)(delayed(ut.cz_fidelity_log_optimize)(args_indep, *arg_select)
                                            for args_indep in x0_vec)
print(f'\nf_noise (dim={len(hspace_select)})  = [')
for i in range(0, len(f_noise), 4):
    print(', '.join(map(str, np.round(f_noise[i:i+4], 8).tolist())), ',')
print(']')